In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score


In [2]:
# Load the dataset
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

# Convert to pandas DataFrames
train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

In [3]:
from transformers import AutoTokenizer

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')


In [4]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,  # Make sure this is False
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }



In [5]:
MAX_LEN = 128
BATCH_SIZE = 32

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    labels=train_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    labels=val_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    labels=test_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [6]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()
        
        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)
        
        self.lstm = nn.LSTM(embedding_dim,
                            hidden_dim,
                            num_layers=n_layers,
                            bidirectional=bidirectional,
                            batch_first=True,
                            dropout=dropout)
        
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        packed_output, (hidden, cell) = self.lstm(embedded)
        
        # Concatenate the final forward and backward hidden states
        if self.lstm.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
        
        output = self.fc(hidden)
        
        return output


In [7]:
embedding_dim = 128
hidden_dim = 256
output_dim = 1  # Binary classification
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)


In [8]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


In [9]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = criterion.to(device)


In [10]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        labels = d["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids)
        preds = outputs.squeeze()
        loss = criterion(preds, labels.float())

        probs = torch.sigmoid(preds)
        preds_cls = torch.round(probs)

        correct_predictions += torch.sum(preds_cls == labels)
        losses.append(loss.item())

        # Detach tensors before converting to NumPy
        all_labels.extend(labels.cpu().detach().numpy())
        all_preds.extend(preds_cls.cpu().detach().numpy())

        loss.backward()
        optimizer.step()

    # Calculate metrics
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    accuracy = correct_predictions.double() / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1

def eval_model(model, data_loader, criterion, device):
    model.eval()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            labels = d["labels"].to(device)

            outputs = model(input_ids)
            preds = outputs.squeeze()
            loss = criterion(preds, labels.float())

            probs = torch.sigmoid(preds)
            preds_cls = torch.round(probs)

            correct_predictions += torch.sum(preds_cls == labels)
            losses.append(loss.item())

            # Detach tensors before converting to NumPy
            all_labels.extend(labels.cpu().detach().numpy())
            all_preds.extend(preds_cls.cpu().detach().numpy())

    # Calculate metrics
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    accuracy = correct_predictions.double() / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1


In [11]:
EPOCHS = 5

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    
    train_acc, train_loss, train_prec, train_rec, train_f1 = train_epoch(
        model, train_loader, optimizer, criterion, device)
    
    val_acc, val_loss, val_prec, val_rec, val_f1 = eval_model(
        model, val_loader, criterion, device)
    
    print(f'Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, '
          f'Precision: {train_prec:.4f}, Recall: {train_rec:.4f}, F1 Score: {train_f1:.4f}')
    
    print(f'Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, '
          f'Precision: {val_prec:.4f}, Recall: {val_rec:.4f}, F1 Score: {val_f1:.4f}')


Epoch 1/5
Train Loss: 0.6873, Accuracy: 0.5468, Precision: 0.5450, Recall: 0.5660, F1 Score: 0.5553
Val Loss: 0.7027, Accuracy: 0.5000, Precision: 0.5000, Recall: 1.0000, F1 Score: 0.6667
Epoch 2/5
Train Loss: 0.6788, Accuracy: 0.5481, Precision: 0.5433, Recall: 0.6035, F1 Score: 0.5718
Val Loss: 0.6726, Accuracy: 0.6107, Precision: 0.5870, Recall: 0.7467, F1 Score: 0.6573
Epoch 3/5
Train Loss: 0.6006, Accuracy: 0.6818, Precision: 0.6821, Recall: 0.6811, F1 Score: 0.6816
Val Loss: 0.6423, Accuracy: 0.6698, Precision: 0.6259, Recall: 0.8443, F1 Score: 0.7188
Epoch 4/5
Train Loss: 0.4581, Accuracy: 0.7905, Precision: 0.7964, Recall: 0.7805, F1 Score: 0.7884
Val Loss: 0.5508, Accuracy: 0.7242, Precision: 0.7043, Recall: 0.7730, F1 Score: 0.7370
Epoch 5/5
Train Loss: 0.3363, Accuracy: 0.8630, Precision: 0.8736, Recall: 0.8488, F1 Score: 0.8610
Val Loss: 0.6181, Accuracy: 0.7364, Precision: 0.7551, Recall: 0.6998, F1 Score: 0.7264


In [12]:
test_acc, test_loss, test_prec, test_rec, test_f1 = eval_model(
    model, test_loader, criterion, device)

print(f'Test Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}, '
      f'Precision: {test_prec:.4f}, Recall: {test_rec:.4f}, F1 Score: {test_f1:.4f}')


Test Loss: 0.5990, Accuracy: 0.7430, Precision: 0.7715, Recall: 0.6904, F1 Score: 0.7287
